# 03회 AICE Associate 모의고사 (다중 분류)
**시나리오: 반도체 공정 불량 등급 예측**

제조 공정 센서 데이터로 제품의 **품질 등급(정상/경미/심각)** 을 예측합니다.
이번 회차 포인트는 **중복 제거, 자료형 변환, 최빈값 채우기, LabelEncoder, softmax**예요.

| 컬럼 | 설명 |
|---|---|
| lot_id | 로트 번호 |
| line | 생산 라인 (A/B/C) |
| shift | 근무조 (주간/야간) |
| temp | 공정 온도 |
| pressure | 압력 (일부가 문자 '-'로 기록됨) |
| humidity | 습도 |
| vibration | 진동 |
| operator_exp | 작업자 경력(년) |
| material | 원자재 공급사 (결측 있음) |
| **grade** | **품질 등급 (타깃: 정상/경미/심각)** |

## 0. 준비 (문제 아님, 실행만 하세요)

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(23)
n = 3000
line = rng.choice(['A','B','C'], n, p=[0.4,0.35,0.25])
shift = rng.choice(['주간','야간'], n, p=[0.6,0.4])
temp = np.round(rng.normal(350, 12, n), 1)
pres = np.round(rng.normal(2.0, 0.25, n), 2)
hum = np.round(rng.normal(45, 8, n), 1)
vib = np.round(rng.gamma(2, 0.5, n), 2)
exp = rng.integers(0, 20, n)
mat = rng.choice(['S사','K사','H사'], n, p=[0.5,0.3,0.2])
risk = (0.08*np.abs(temp-350) + 2.5*np.abs(pres-2.0) + 1.2*vib - 0.08*exp
        + 0.8*(shift=='야간') + 0.6*(line=='C') + 0.5*(mat=='H사') + rng.normal(0, 0.7, n))
grade = np.where(risk > 3.6, '심각', np.where(risk > 2.3, '경미', '정상'))
df = pd.DataFrame({'lot_id': [f'L{i:05d}' for i in range(n)], 'line': line, 'shift': shift,
    'temp': temp, 'pressure': pres.astype(str), 'humidity': hum, 'vibration': vib,
    'operator_exp': exp, 'material': mat, 'grade': grade})
df.loc[rng.choice(n, 25, replace=False), 'pressure'] = '-'
df.loc[rng.choice(n, 90, replace=False), 'material'] = np.nan
df = pd.concat([df, df.sample(60, random_state=1)], ignore_index=True)   # 중복 행 60개
df.to_csv('process_data.csv', index=False)
print('process_data.csv 생성 완료')

## 1. 라이브러리 임포트

pandas(`pd`), numpy(`np`), scikit-learn(`sk`), seaborn(`sns`), matplotlib.pyplot(`plt`)을 별칭(alias)으로 임포트하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
import pandas as pd
import numpy as np
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
```


</details>

## 2. 데이터 불러오기와 확인

`process_data.csv`를 **`df`** 로 읽고 `info()`로 확인하세요.

숫자여야 할 것 같은데 **문자(object)로 되어 있는 컬럼**은 무엇인가요?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df = pd.read_csv('process_data.csv')
df.info()
```

답: **pressure**. 숫자 사이에 `'-'` 같은 문자가 섞여 있으면 열 전체가 문자로 읽혀요. 6번에서 고칩니다.

(pandas 버전에 따라 object 대신 str이나 string으로 보일 수도 있어요. 같은 뜻이에요.)
</details>

## 3. 시각화 (1) 타깃 분포

`grade`의 값별 개수를 **countplot** 으로 그리고 `value_counts()`로 숫자도 확인하세요. 가장 많은 등급은?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.countplot(data=df, x='grade')
plt.show()
df['grade'].value_counts()
```

답: **정상** (1,628개). 그래프에 한글이 네모로 깨지면 정상이에요 (폰트 문제). 시험에선 신경 안 써도 되고, 숫자는 value_counts로 확인하면 돼요.
</details>

## 4. 중복 행 제거

완전히 똑같은 **중복 행**이 몇 개인지 확인하고, 제거한 결과를 **`df1`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
print(df.duplicated().sum())    # 중복 행 개수
df1 = df.drop_duplicates()
print(df1.shape)
```

답: 중복 **60개**, 남은 행 **(3000, 10)**.

`duplicated()`는 중복이면 True, `.sum()`하면 True 개수가 나와요.
</details>

## 5. 불필요한 컬럼 삭제

`df1`에서 `lot_id`를 삭제해 **`df2`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df2 = df1.drop(columns=['lot_id'])
```


</details>

## 6. 자료형 변환

`pressure`의 `'-'` 값을 결측치로 바꾸면서 **숫자형(float)으로 변환**하고, 그 결측 행은 **삭제**해 `df2`에 반영하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df2 = df2.copy()
df2['pressure'] = pd.to_numeric(df2['pressure'], errors='coerce')   # 숫자로 못 바꾸는 건 NaN으로
df2 = df2.dropna(subset=['pressure'])
print(df2['pressure'].dtype, len(df2))
```

답: float64, **2975행**.

`errors='coerce'`가 핵심이에요. 숫자로 못 바꾸는 값을 에러 대신 NaN으로 만들어 줘요. 다른 방법: `df2['pressure'].replace('-', np.nan).astype(float)`
</details>

## 7. 결측치 처리 (최빈값)

`material`의 결측치를 **최빈값(가장 많이 나온 값)** 으로 채우세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
print(df2['material'].mode()[0])    # 최빈값
df2['material'] = df2['material'].fillna(df2['material'].mode()[0])
df2.isnull().sum()
```

답: 최빈값은 **S사**.

문자 열은 평균을 못 내니까 최빈값으로 채워요. `mode()`는 결과가 여러 개일 수 있어서 **`[0]`** 으로 첫 번째를 꺼내야 해요. 이거 빼먹으면 틀려요!
</details>

## 8. 시각화 (2) 상관관계

숫자 컬럼만으로 상관계수 heatmap을 그리세요. **`vibration`과 상관이 가장 큰 다른 변수**는 있나요?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.heatmap(df2.corr(numeric_only=True), annot=True, fmt='.2f')
plt.show()
```

답: 없어요. 전부 0에 가까워요. 센서끼리 서로 독립적이라는 뜻이에요.

`numeric_only=True`를 넣으면 문자 열은 알아서 빼고 계산해요.
</details>

## 9. 타깃 라벨 인코딩

타깃 `grade`(문자)를 **LabelEncoder** 로 숫자로 바꾸세요. 어떤 문자가 어떤 숫자가 됐는지도 확인하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df2['grade'] = le.fit_transform(df2['grade'])
print(le.classes_)    # 0, 1, 2 순서대로 원래 이름
```

답: `['경미' '심각' '정상']` → 경미=0, 심각=1, 정상=2 (가나다순).

**타깃은 LabelEncoder, X의 문자 열은 get_dummies** 가 기본 공식이에요.
</details>

## 10. 원핫 인코딩 + 데이터 분리

`line`, `shift`, `material`을 원핫 인코딩(`drop_first=True`)해 **`df_preset`** 에 저장하고, `grade`를 y로 `test_size=0.2, stratify=y, random_state=42` 분리하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.model_selection import train_test_split

df_preset = pd.get_dummies(df2, columns=['line', 'shift', 'material'], drop_first=True)

X = df_preset.drop('grade', axis=1)
y = df_preset['grade']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape)
```

답: **(2380, 10)**. 분류라서 stratify 사용!
</details>

## 11. 스케일링

StandardScaler로 X_train, X_test를 스케일링하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
```


</details>

## 12. 머신러닝 모델 + 변수 중요도

**RandomForestClassifier**(`n_estimators=100, random_state=42`)를 `rf`로 학습하고, **feature_importances_** 로 가장 중요한 변수를 찾으세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp)
```

답: **vibration** 이 1등이에요.

스케일링 후 X_train은 컬럼 이름이 없는 배열이라 `index=X.columns`로 이름을 붙여요. 트리 계열 모델(DT, RF)만 feature_importances_가 있어요.
</details>

## 13. 다중 분류 평가

`rf`의 **accuracy, confusion_matrix, classification_report** 를 출력하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pred = rf.predict(X_test)
print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=le.classes_))
```

confusion_matrix에서 **대각선**이 맞힌 개수, 나머지는 헷갈린 개수예요. 행이 실제, 열이 예측.
</details>

## 14. 딥러닝 다중 분류

DNN을 만들고 학습한 뒤, X_test 첫 5개의 **예측 클래스**를 출력하세요.
- Dense(64, relu) → Dropout(0.3) → Dense(32, relu) → **다중 분류 출력층**
- adam, loss는 y가 0/1/2 정수인 것에 맞게, metrics `['accuracy']`
- EarlyStopping(`patience=5`), ModelCheckpoint(`'best_model.keras'`, `save_best_only=True`)
- epochs=50, batch_size=32, `validation_data=(X_test, y_test)`

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')        # 클래스 3개
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=5)
mc = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=32,
                    validation_data=(X_test, y_test), callbacks=[es, mc])

proba = model.predict(X_test[:5])    # 클래스별 확률 3개씩
print(np.argmax(proba, axis=1))      # 확률이 가장 큰 클래스 번호
```

다중 분류: **`Dense(클래스수, softmax)`** + **`sparse_categorical_crossentropy`** (y가 0,1,2 정수일 때).

딥러닝 predict는 확률을 줘요. 다중 분류는 **`np.argmax(..., axis=1)`**, 이진 분류는 **`(proba > 0.5).astype(int)`** 로 클래스를 만들어요.
</details>

---
# 🏁 마무리 체크리스트
- 막히면 `함수이름?` 실행 또는 괄호 안에서 **Shift + Tab**
- `train_test_split` 반환 순서: **X_train, X_test, y_train, y_test**
- test 데이터는 **transform만** (fit 금지)
- 출력층과 loss 짝꿍 확인 (이진 sigmoid+binary_crossentropy / 다중 softmax+sparse_categorical_crossentropy / 회귀 Dense(1)+mse)
- 문제에서 정한 **변수명** 그대로 쓰기
- 다 풀었으면 정답 안 보고 **60분 타이머** 맞춰 한 번 더!